In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter


# =========================================================
# Global settings
# =========================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib").setLevel(logging.ERROR)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


sns.set_style("white")

# =========================

# =========================

BASE_MULTI = "fig3/code_save_6_7/MultiModal_Run_update_cox_fishing_0607_full_final/ecg_soma_snp_fundus_20260608_094805_longitudinal"

BASE_SINGLE = {
    "snp":  "code_02_02_save/snp_20260202_1432/explanations",
    "soma": "code_02_02_save/soma_20260202_2006/explanations",
    "ecg":  "code_02_02_save/ecg_20260203_1003/explanations",
    "fundus": "code_02_02_save/fundus_20251229_1519/explanations"
}

# =========================

# =========================

def safe_flat(arr):
    arr = np.asarray(arr)
    return np.abs(arr[np.isfinite(arr)].flatten())

# =========================
# MULTIMODAL
# =========================

def load_multi(endpoint):

    data = {}

    # SNP
    df = pd.read_csv(f"{BASE_MULTI}/{endpoint}_importance_snp.csv")
    data["SNP"] = safe_flat(df.select_dtypes(float).values)

    # Soma
    df = pd.read_csv(f"{BASE_MULTI}/{endpoint}_importance_soma.csv")
    data["Soma"] = safe_flat(df.select_dtypes(float).values)

    # ECG
    ecg_dir = f"{BASE_MULTI}/{endpoint}_importance_ecg"
    vals = []
    for f in os.listdir(ecg_dir):
        if f.endswith(".txt"):
            vals.append(np.loadtxt(os.path.join(ecg_dir, f)))
    data["ECG"] = safe_flat(np.concatenate(vals)) if vals else np.array([])

    # Fundus
    vals = []
    for f in os.listdir(BASE_MULTI):
        if endpoint in f and f.startswith("gradcam"):
            vals.append(np.loadtxt(os.path.join(BASE_MULTI, f)))
    data["Fundus"] = safe_flat(np.concatenate(vals)) if vals else np.array([])

    return data

# =========================
# SINGLE MODALITY
# =========================

def load_single(endpoint):

    data = {}

    # SNP
    snp_path = f"{BASE_SINGLE['snp']}/{endpoint}_snp_TR_best_IG.csv"
    df = pd.read_csv(snp_path)
    data["SNP"] = safe_flat(df["ig_abs"].values)

    # Soma
    soma_path = f"{BASE_SINGLE['soma']}/{endpoint}_soma_TR_best_IG.csv"
    df = pd.read_csv(soma_path)
    data["Soma"] = safe_flat(df["ig_abs"].values)

    # ECG
    ecg_dir = BASE_SINGLE["ecg"]
    vals = []
    for root, _, files in os.walk(f"code_3_3_save/ECG_single/{endpoint}_ecg_transformer_h256_l3_ig_ecg_per_sample/"):
        for f in files:
            if f.endswith("_ig_ecg.txt"):
                vals.append(np.loadtxt(os.path.join(root, f)))
    data["ECG"] = safe_flat(np.concatenate(vals)) if vals else np.array([])

    # Fundus
    fundus_dir = BASE_SINGLE["fundus"]
    vals = []
    for f in os.listdir(fundus_dir):
        if endpoint in f and f.startswith("gradcam"):
            vals.append(np.loadtxt(os.path.join(fundus_dir, f)))
    data["Fundus"] = safe_flat(np.concatenate(vals)) if vals else np.array([])

    return data

# =========================

# =========================

def plot_compare(endpoint):

    multi = load_multi(endpoint)
    single = load_single(endpoint)

    rows = []

    for mod in ["SNP","Soma","ECG","Fundus"]:
        for v in single[mod]:
            rows.append([mod, "Single", v])
        for v in multi[mod]:
            rows.append([mod, "Multimodal", v])

    df = pd.DataFrame(rows, columns=["Modality","Type","Value"])

    
    df = df[df["Value"] > 0]
    df["Value"] = np.clip(df["Value"], 1e-8, df["Value"].quantile(0.995))

    plt.figure(figsize=(7,5))

    sns.boxplot(
    data=df,
    x="Modality",
    y="Value",
    hue="Type",
    showfliers=False,
    palette={
        "Single": "#999999",
        "Multimodal": "#C44E52"
    })
    plt.yscale("log")

    plt.title(f"{endpoint.upper()} IG / Grad-CAM comparison")
    plt.ylabel("Attribution (log scale)")

    sns.despine()
    plt.tight_layout()

    plt.savefig(f"figs6_final/IG_compare_{endpoint}.pdf", dpi=300, transparent=True)
    plt.show()

# =========================
# RUNs
# =========================

if __name__ == "__main__":

    plot_compare("ICVD")
    #plot_compare("istroke")

    print("Done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================

BASE = "fig3/code_save_6_7/MultiModal_Run_update_cox_fishing_0607_full_final/ecg_soma_snp_fundus_20260608_094805_longitudinal"
ENDPOINT = "ICVD"

OUT_DIR = "figs6_final/fig_single_vs_multi_pearson_pid"
os.makedirs(OUT_DIR, exist_ok=True)

MODS = ["ecg", "soma", "snp"]

# =========================================================
# RISK
# =========================================================

def load_risk():
    df = pd.read_csv("fig3/longitudinal/ecg_soma_snp_fundus_20251230_173503_longitudinal/ICVD_pred.csv")

    if "pid" not in df.columns:
        for c in df.columns:
            if "id" in c.lower():
                df = df.rename(columns={c: "pid"})
                break

    risk_col = [c for c in df.columns if "risk" in c.lower() or "pred" in c.lower()][0]

    df = df[["pid", risk_col]]
    df.columns = ["pid", "risk"]

    df["pid"] = df["pid"].astype(int)
    df["risk"] = df["risk"].astype(float)

    return df


# =========================================================
# SAFE FLATTEN
# =========================================================

def safe_flat(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    return np.abs(x).ravel()


# =========================================================
# LOAD SINGLE / MULTI (PID-level dict)
# =========================================================

def load_single(mod):

    if mod == "soma":
        df = pd.read_csv(f"code_02_02_save/soma_20260202_2006/explanations/{ENDPOINT}_soma_TR_best_IG.csv")
        return df

    if mod == "snp":
        df = pd.read_csv(f"code_02_02_save/snp_20260202_1432/explanations/{ENDPOINT}_snp_TR_best_IG.csv")
        return df

    if mod == "ecg":
        base = f"code_3_3_save/ECG_single/{ENDPOINT}_ecg_transformer_h256_l3_ig_ecg_per_sample"
        out = {}
        for r, _, fs in os.walk(base):
            for f in fs:
                if f.endswith("_ig_ecg.txt"):
                    pid = int(f.split("_")[0])
                    out[pid] = np.loadtxt(os.path.join(r, f))
        return out


def load_multi(mod):

    if mod == "soma":
        df = pd.read_csv(f"{BASE}/{ENDPOINT}_importance_soma.csv")
        return df

    if mod == "snp":
        df = pd.read_csv(f"{BASE}/{ENDPOINT}_importance_snp.csv")
        return df

    if mod == "ecg":
        base = f"{BASE}/{ENDPOINT}_importance_ecg"
        out = {}
        for f in os.listdir(base):
            if f.endswith(".txt"):
                pid = int(f.split("_")[0])
                out[pid] = np.loadtxt(os.path.join(base, f))
        return out


# =========================================================
# BUILD PID VECTOR (IMPORTANT FIX)
# =========================================================

def build_pid_vector(df, mod):

    out = {}

    if mod in ["soma", "snp"]:

        # df: pid, feature, ig_abs
        for pid in df["pid"].unique():

            v = df[df["pid"] == pid]["ig_abs"].values
            out[int(pid)] = safe_flat(v)

        return out

    # ECG / others already dict
    return df


# =========================================================
# PEARSON PER PID
# =========================================================

def pearson(a, b):

    a = safe_flat(a)
    b = safe_flat(b)

    n = min(len(a), len(b))
    a, b = a[:n], b[:n]

    if n < 20:
        return np.nan

    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan

    return np.corrcoef(a, b)[0, 1]


# =========================================================
# MAIN CORE
# =========================================================

def compute(mod, risk_df):

    single = load_single(mod)
    multi = load_multi(mod)

    # =========================
    # convert SNP/SOMA table → dict
    # =========================
    if mod in ["soma", "snp"]:

        single = build_pid_vector(single, mod)
        multi = build_pid_vector(multi, mod)

    common_pids = set(single.keys()) & set(multi.keys())

    rows = []

    for pid in common_pids:

        r = pearson(single[pid], multi[pid])

        rows.append([pid, r])

    df = pd.DataFrame(rows, columns=["pid", "pearson"])

    # merge risk
    risk_map = dict(zip(risk_df.pid, risk_df.risk))
    df["risk"] = df["pid"].map(risk_map)

    df = df.dropna()

    # percentile
    df = df.sort_values("risk")
    df["pct"] = np.linspace(0, 100, len(df))

    df["group"] = np.where(df["pct"] >= 50, "High", "Low")

    return df


# =========================================================
# PLOT (WITH MEAN LINE)
# =========================================================

def plot(df, mod):

    low = df[df.group == "Low"]
    high = df[df.group == "High"]

    low_mean = low.pearson.mean()
    high_mean = high.pearson.mean()

    plt.figure(figsize=(5.5,4))

    plt.scatter(low.pct, low.pearson, s=18, c="#4C72B0")
    plt.scatter(high.pct, high.pearson, s=18, c="#C44E52")

    plt.axvline(50, ls="--", c="gray")

    # ===== mean lines =====
    plt.hlines(low_mean, 0, 50, colors="#4C72B0", linestyles="--", linewidth=2)
    plt.hlines(high_mean, 50, 100, colors="#C44E52", linestyles="--", linewidth=2)

    plt.text(2, low_mean, f"Low mean={low_mean:.3f}", color="#4C72B0")
    plt.text(52, high_mean, f"High mean={high_mean:.3f}", color="#C44E52")

    plt.xlabel("Risk percentile")
    plt.ylabel("Pearson (single vs multimodal IG)")
    plt.title(f"{mod.upper()} ICVD PID-level IG correlation")

    plt.tight_layout()

    out = f"{OUT_DIR}/{mod}_PID_level_pearson.pdf"
    plt.savefig(out, dpi=300, transparent=True)
    plt.close()

    print("saved:", out)


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    risk_df = load_risk()

    for m in MODS:
        df = compute(m, risk_df)
        plot(df, m)

    print("\nDONE")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

# =========================================================
# CONFIG
# =========================================================

BASE = "."
ENDPOINT = "ICVD"

SINGLE_CAM_DIR = os.path.join(
    BASE,
    "code_02_02_save/fundus_20251229_1519/explanations"
)

MULTI_CAM_DIR = os.path.join(
    BASE,
    "fig3/code_save_6_7/"
    "MultiModal_Run_update_cox_fishing_0607_full_final/"
    "ecg_soma_snp_fundus_20260608_094805_longitudinal"
)

MULTI_RISK_CSV = os.path.join(
    BASE,
    "fig3/longitudinal/"
    "ecg_soma_snp_fundus_20251230_173503_longitudinal/"
    "ICVD_pred.csv"
)

OUT_DIR = os.path.join(
    BASE,
    "figs6_final/fig_single_vs_multi_pearson_pid"
)
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_SIZE = 224
EPS = 1e-8

LOW_COLOR = "#4C72B0"
HIGH_COLOR = "#C44E52"

# =========================================================
# HELPERS
# =========================================================

def minmax_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)

    if not np.isfinite(xmin) or not np.isfinite(xmax):
        return np.zeros_like(x, dtype=np.float32)

    if xmax - xmin < EPS:
        return np.zeros_like(x, dtype=np.float32)

    return (x - xmin) / (xmax - xmin + EPS)


def safe_pearson(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()

    mask = np.isfinite(a) & np.isfinite(b)
    a = a[mask]
    b = b[mask]

    if len(a) < 3:
        return np.nan

    if np.std(a) < EPS or np.std(b) < EPS:
        return np.nan

    return float(pearsonr(a, b)[0])


def load_cam(fp):
    cam = np.loadtxt(fp)

    if cam.ndim == 1 and cam.size == 196:
        cam = cam.reshape(14, 14)

    if cam.ndim == 1:
        side = int(np.sqrt(cam.size))
        if side * side == cam.size:
            cam = cam.reshape(side, side)
        else:
            raise ValueError(f"Cannot reshape CAM: {fp}, size={cam.size}")

    return cam.astype(np.float32)


def load_and_resize_cam(fp, normalize_each=True, target_size=TARGET_SIZE):
    cam = load_cam(fp)

    if normalize_each:
        cam = minmax_normalize(cam)

    cam = np.array(
        Image.fromarray(cam).resize(
            (target_size, target_size),
            resample=Image.BILINEAR
        )
    ).astype(np.float32)

    return cam


def build_gradcam_index(cam_dir, endpoint):
    pattern = os.path.join(cam_dir, f"gradcam_*_*_{endpoint}.txt")
    files = sorted(glob.glob(pattern))

    idx = {}
    bad = 0
    duplicate = 0

    for fp in files:
        fn = os.path.basename(fp).replace(".txt", "")
        parts = fn.split("_")

        try:
            pid = int(parts[-2])
        except Exception:
            bad += 1
            continue

        if pid in idx:
            duplicate += 1

        idx[pid] = fp

    print(f"[CAM index] {cam_dir}")
    print(f"  files={len(files)}, indexed={len(idx)}, duplicate={duplicate}, bad={bad}")

    return idx


def load_risk():
    df = pd.read_csv(MULTI_RISK_CSV)

    if "pid" not in df.columns:
        for c in df.columns:
            if "id" in c.lower():
                df = df.rename(columns={c: "pid"})
                break

    risk_col = [c for c in df.columns if "risk" in c.lower() or "pred" in c.lower()][0]

    df = df[["pid", risk_col]].copy()
    df.columns = ["pid", "risk"]

    df["pid"] = df["pid"].astype(int)
    df["risk"] = df["risk"].astype(float)

    return df

# =========================================================
# COMPUTE FUNDUS PER-PID PEARSON
# =========================================================

def compute_fundus(risk_df):
    single_idx = build_gradcam_index(SINGLE_CAM_DIR, ENDPOINT)
    multi_idx = build_gradcam_index(MULTI_CAM_DIR, ENDPOINT)

    common_pids = sorted(set(single_idx.keys()) & set(multi_idx.keys()))
    risk_map = dict(zip(risk_df["pid"], risk_df["risk"]))

    print("common CAM pid n =", len(common_pids))

    rows = []

    for pid in common_pids:
        if pid not in risk_map:
            continue

        try:
            single_cam = load_and_resize_cam(single_idx[pid], normalize_each=True)
            multi_cam = load_and_resize_cam(multi_idx[pid], normalize_each=True)

            single_cam = minmax_normalize(single_cam)
            multi_cam = minmax_normalize(multi_cam)

            r = safe_pearson(single_cam, multi_cam)

            rows.append({
                "pid": int(pid),
                "pearson": r,
                "risk": float(risk_map[pid]),
                "single_cam_file": single_idx[pid],
                "multi_cam_file": multi_idx[pid],
            })

        except Exception as e:
            print(f"[WARN] failed pid={pid}: {e}")

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["pearson", "risk"]).copy()

    df = df.sort_values("risk").reset_index(drop=True)
    df["pct"] = np.linspace(0, 100, len(df))
    df["group"] = np.where(df["pct"] >= 50, "High", "Low")

    return df

# =========================================================
# PLOT: SAME STYLE AS ECG/SNP/SOMA CODE
# =========================================================

def plot(df):
    low = df[df.group == "Low"]
    high = df[df.group == "High"]

    low_mean = low.pearson.mean()
    high_mean = high.pearson.mean()

    plt.figure(figsize=(5.5, 4))

    plt.scatter(low.pct, low.pearson, s=18, c=LOW_COLOR)
    plt.scatter(high.pct, high.pearson, s=18, c=HIGH_COLOR)

    plt.axvline(50, ls="--", c="gray")

    plt.hlines(low_mean, 0, 50, colors=LOW_COLOR, linestyles="--", linewidth=2)
    plt.hlines(high_mean, 50, 100, colors=HIGH_COLOR, linestyles="--", linewidth=2)

    plt.text(2, low_mean, f"Low mean={low_mean:.3f}", color=LOW_COLOR)
    plt.text(52, high_mean, f"High mean={high_mean:.3f}", color=HIGH_COLOR)

    plt.xlabel("Risk percentile")
    plt.ylabel("Pearson (single vs multimodal IG)")
    plt.title("FUNDUS ICVD PID-level IG correlation")

    plt.tight_layout()

    out = os.path.join(OUT_DIR, "fundus_PID_level_pearson.pdf")
    plt.savefig(out, dpi=300, transparent=True)
    plt.close()

    print("saved:", out)

# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":
    risk_df = load_risk()

    df = compute_fundus(risk_df)

    out_csv = os.path.join(
        OUT_DIR,
        f"{ENDPOINT}_fundus_per_sample_single_vs_multimodal_gradcam_pearson.csv"
    )
    df.to_csv(out_csv, index=False)
    print("saved:", out_csv)

    plot(df)

    print("\nDONE")